In [149]:
# Autoreload para refletir mudanças no config sem reiniciar kernel
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [150]:
# Célula 1.5 — garante instalação no kernel atual
import subprocess
import sys
from pathlib import Path

project_root = Path("..").resolve()

result = subprocess.run(
    ["uv", "pip", "install", "-e", str(project_root), "-q"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
else:
    print("churn_telecom instalado com sucesso.")

churn_telecom instalado com sucesso.


In [151]:
# Célula 2 — stdlib
import logging

# terceiros
import pandas as pd

In [152]:
# Célula 3 — módulo interno
from churn_telecom.config import (
    COLS_CAT,
    COLS_ID,
    COLS_NUM,
    COLS_POS,
    DATA_INTERIM,
    DATA_ROWS,
    LABEL_COL,
    RANDOM_STATE,
    TARGET,
    get_logger,
    to_snake_case,
)

In [153]:
logger = get_logger("1.01_vab_data_cleaning")
logger.info("Iniciando análise de limpeza de dados | 1.01_vab_data_cleaning")

Iniciando análise de limpeza de dados | 1.01_vab_data_cleaning


In [154]:
# Célula 5 — carrega dataset tipado (saída do 0.01)
df_raw = pd.read_parquet(DATA_INTERIM / "telco_typed.parquet")
logger.info("Dataset carregado | shape=%s", df_raw.shape)
df_raw.head()

Dataset carregado | shape=(7043, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


In [155]:
# Célula 6 — cópia segura — nunca modificar o raw
df = df_raw.copy()

# snapshot antes da limpeza — rastreia o delta ao final
shape_antes = df.shape
nulos_antes = df.isnull().sum().sum()
logger.info("SNAPSHOT ANTES | shape=%s | nulos_totais=%d", shape_antes, nulos_antes)

SNAPSHOT ANTES | shape=(7043, 20) | nulos_totais=11


In [156]:
# Célula 7 — padronização de nomes de colunas para snake_case
# to_snake_case importado do config.py — única fonte de verdade
df.columns = [to_snake_case(col) for col in df.columns]
logger.info("Colunas padronizadas para snake_case | total=%d", len(df.columns))

Colunas padronizadas para snake_case | total=20


In [157]:
# Célula 8 — validação de shape
# DATA_ROWS definido no config.py (7043)
delta = abs(df.shape[0] - DATA_ROWS)
if delta <= 5:
    logger.info(
        "Shape OK | linhas=%d (esperado=%d | delta=%d)",
        df.shape[0], DATA_ROWS, delta,
    )
else:
    logger.warning(
        "Shape inesperado | linhas=%d | esperado=%d | delta=%d",
        df.shape[0], DATA_ROWS, delta,
    )

Shape OK | linhas=7043 (esperado=7043 | delta=0)


In [158]:
# Célula 9 — validação de schema
# converte constantes para snake_case para comparação consistente
expected_cols = {
    to_snake_case(c)
    for c in COLS_ID + COLS_NUM + COLS_CAT + COLS_POS + [TARGET, LABEL_COL]
}
current_cols = set(df.columns)

missing_cols = expected_cols - current_cols
extra_cols   = current_cols - expected_cols

if missing_cols:
    logger.warning("Colunas ausentes no df : %s", sorted(missing_cols))
if extra_cols:
    logger.warning("Colunas extras no df   : %s", sorted(extra_cols))
if not missing_cols and not extra_cols:
    logger.info("Schema validado — todas as colunas esperadas presentes.")

Colunas ausentes no df : ['churn_label', 'churn_reason', 'churn_score', 'city', 'cltv', 'count', 'country', 'customerid', 'lat_long', 'latitude', 'longitude', 'state', 'zip_code']


In [159]:
# Célula 10 — remoção de colunas de ID e geo
# sem poder preditivo — apenas identificadores e localização
cols_id_snake = [to_snake_case(c) for c in COLS_ID]
cols_id_present = [c for c in cols_id_snake if c in df.columns]

df.drop(columns=cols_id_present, inplace=True)
logger.info(
    "Colunas ID/Geo removidas (%d) | %s",
    len(cols_id_present), cols_id_present,
)

Colunas ID/Geo removidas (0) | []


In [160]:
# Célula 11 — remoção de colunas com data leakage
# COLS_POS: geradas após o evento de churn — nunca usar como features
cols_pos_snake = [to_snake_case(c) for c in COLS_POS]
cols_pos_present = [c for c in cols_pos_snake if c in df.columns]

df.drop(columns=cols_pos_present, inplace=True)
logger.warning(
    "Leakage removido (%d colunas) | %s",
    len(cols_pos_present), cols_pos_present,
)

Leakage removido (0 colunas) | []


In [161]:
# Célula 12 — remoção de label_col (versão string do target — redundante)
label_snake = to_snake_case(LABEL_COL)
if label_snake in df.columns:
    df.drop(columns=[label_snake], inplace=True)
    logger.info("label_col removida: %s", label_snake)

In [162]:
# Célula 13 — garantia e tipagem do target
target_snake = to_snake_case(TARGET)

# fallback: busca coluna com 'churn' no nome caso target não encontrado
if target_snake not in df.columns:
    candidates = [c for c in df.columns if "churn" in c]
    if candidates:
        df.rename(columns={candidates[0]: target_snake}, inplace=True)
        logger.warning(
            "TARGET renomeado: %s → %s", candidates[0], target_snake,
        )
    else:
        logger.error("TARGET '%s' não encontrado no dataset.", target_snake)

df[target_snake] = df[target_snake].astype(int)
logger.info("TARGET '%s' validado como int.", target_snake)

TARGET 'churn_value' validado como int.


In [163]:
# Célula 14 — imputação de nulos em total_charges
# 11 registros com tenure=0 (recém-chegados sem cobrança acumulada)
# decisão (literatura): mediana — preserva distribuição melhor que 0.0
total_charges_col = to_snake_case("Total Charges")

if total_charges_col in df.columns:
    n_nulos_tc = df[total_charges_col].isna().sum()

    # validação da hipótese: confirma que todos os nulos da coluna total_charges têm tenure=0
    tenure_col = to_snake_case("Tenure Months")
    if n_nulos_tc > 0 and tenure_col in df.columns:
        tenure_dos_nulos = df.loc[df[total_charges_col].isna(), tenure_col].unique()
        logger.info(
            "%s | nulos=%d | tenure dos nulos=%s | hipótese tenure=0: %s",
            total_charges_col, n_nulos_tc, tenure_dos_nulos,
            all(tenure_dos_nulos == 0),
        )

    mediana_tc = df[total_charges_col].median()
    df[total_charges_col] = df[total_charges_col].fillna(mediana_tc)

    logger.info(
        "%s | %d nulos imputados com mediana=%.2f",
        total_charges_col, n_nulos_tc, mediana_tc,
    )

total_charges | nulos=11 | tenure dos nulos=[0] | hipótese tenure=0: True
total_charges | 11 nulos imputados com mediana=1397.47


In [164]:
# Célula 15 — normalização semântica: "No internet service" → "No"
# reduz cardinalidade e elimina redundância semântica
# "No internet service" e "No" têm o mesmo significado preditivo
NO_SERVICE_COLS = [
    to_snake_case(c) for c in [
        "Online Security", "Online Backup", "Device Protection",
        "Tech Support", "Streaming TV", "Streaming Movies",
    ]
]

for col in NO_SERVICE_COLS:
    if col not in df.columns:
        continue

    # atenção: comparar com o valor real da coluna, não com snake_case
    n_afetados = (df[col] == "No internet service").sum()

    if n_afetados > 0:
        df[col] = df[col].replace({"No internet service": "No"})
        logger.info(
            "%s | 'No internet service' → 'No' | registros=%d",
            col, n_afetados,
        )

logger.info("Normalização semântica concluída.")

online_security | 'No internet service' → 'No' | registros=1526
online_backup | 'No internet service' → 'No' | registros=1526
device_protection | 'No internet service' → 'No' | registros=1526
tech_support | 'No internet service' → 'No' | registros=1526
streaming_tv | 'No internet service' → 'No' | registros=1526
streaming_movies | 'No internet service' → 'No' | registros=1526
Normalização semântica concluída.


In [165]:
# Célula 16 — correção de inconsistência lógica
# regra de negócio: cliente sem internet não pode ter serviços de internet ativos
# comparação com o valor real da coluna: "No" (não snake_case)
internet_col = to_snake_case("Internet Service")

if internet_col in df.columns:
    mask_sem_internet = df[internet_col] == "No"

    for col in NO_SERVICE_COLS:
        if col not in df.columns:
            continue

        n_inconsistentes = (mask_sem_internet & (df[col] != "No")).sum()

        if n_inconsistentes > 0:
            df.loc[mask_sem_internet, col] = "No"
            logger.warning(
                "%s | %d registros inconsistentes corrigidos "
                "(sem internet mas com serviço ativo)",
                col, n_inconsistentes,
            )

    logger.info("Correção de inconsistências lógicas concluída.")

Correção de inconsistências lógicas concluída.


In [166]:
# Célula 19 — validação de ranges numéricos
# detecta anomalias que escapariam de análises simples de nulos
RANGE_RULES = {
    to_snake_case("Tenure Months"):    (0,   72),
    to_snake_case("Monthly Charges"):  (0,  200),
    to_snake_case("Total Charges"):    (0, 10_000),
}

for col, (min_v, max_v) in RANGE_RULES.items():
    if col not in df.columns:
        continue

    n_invalidos = ((df[col] < min_v) | (df[col] > max_v)).sum()

    if n_invalidos > 0:
        logger.warning(
            "%s | %d valores fora do range [%.0f, %.0f]",
            col, n_invalidos, min_v, max_v,
        )
    else:
        logger.info("%s | range OK [%.0f, %.0f]", col, min_v, max_v)

tenure_months | range OK [0, 72]
monthly_charges | range OK [0, 200]
total_charges | range OK [0, 10000]


In [167]:
# Célula 20 — remoção de duplicatas
n_dup = df.duplicated().sum()

if n_dup > 0:
    df.drop_duplicates(inplace=True)
    logger.warning("Duplicatas removidas: %d", n_dup)
else:
    logger.info("Sem duplicatas encontradas.")

Duplicatas removidas: 22


In [168]:
# Célula 22 — resumo da limpeza
shape_depois = df.shape
nulos_depois = df.isnull().sum().sum()

logger.info(
    "RESUMO LIMPEZA | "
    "linhas: %d→%d (delta=%d) | "
    "colunas: %d→%d (removidas=%d) | "
    "nulos: %d→%d",
    shape_antes[0], shape_depois[0], shape_antes[0] - shape_depois[0],
    shape_antes[1], shape_depois[1], shape_antes[1] - shape_depois[1],
    nulos_antes,    nulos_depois,
)

RESUMO LIMPEZA | linhas: 7043→7021 (delta=22) | colunas: 20→20 (removidas=0) | nulos: 11→0


In [169]:
# Célula 23 — persistência e validação pós-save
output_path = DATA_INTERIM / "telco_cleaned.parquet"
df.to_parquet(output_path, index=False)

# validação pós-save: relê o arquivo e compara shape
df_check = pd.read_parquet(output_path)

assert df_check.shape == df.shape, (
    f"Erro na persistência | esperado={df.shape} | lido={df_check.shape}"
)

logger.info(
    "Dataset limpo salvo e validado | path=%s | shape=%s",
    output_path, df.shape,
)
logger.info("1.01 data_cleaning concluído com sucesso.")


Dataset limpo salvo e validado | path=C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final\data\interim\telco_cleaned.parquet | shape=(7021, 20)
1.01 data_cleaning concluído com sucesso.
